#### Install Libraries

In [1]:
%pip install --break-system-packages google-cloud-bigquery pandas numpy scikit-learn matplotlib seaborn --quiet
%pip install google-cloud-bigquery

Note: you may need to restart the kernel to use updated packages.


d:\ML-AI\ai-on-healthcare\.venv\Scripts\python.exe: No module named pip


Note: you may need to restart the kernel to use updated packages.


d:\ML-AI\ai-on-healthcare\.venv\Scripts\python.exe: No module named pip


#### Import libraries

#### Authenticate on google

In [4]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')
from google.cloud import bigquery
project_id = 'ai-on-healthcare'
client = bigquery.Client(project=project_id)

ModuleNotFoundError: No module named 'google.colab'

#### Query MIMIC III Women Mortality Data

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Query to get women patients with mortality information from MIMIC III
query = """
SELECT 
    p.subject_id,
    p.gender,
    p.dob,
    a.hadm_id,
    a.admittime,
    a.dischtime,
    a.admission_type,
    a.diagnosis,
    a.hospital_expire_flag as mortality,
    COUNT(DISTINCT ie.icustay_id) as num_icu_stays,
    AVG(CAST(vs.heart_rate AS FLOAT64)) as avg_heart_rate,
    AVG(CAST(vs.sys_bp AS FLOAT64)) as avg_sys_bp,
    AVG(CAST(vs.dias_bp AS FLOAT64)) as avg_dias_bp,
    AVG(CAST(vs.temp_c AS FLOAT64)) as avg_temp,
    AVG(CAST(vs.resp_rate AS FLOAT64)) as avg_resp_rate
FROM 
    `physionet-data.mimic_iii_clinical.patients` p
JOIN 
    `physionet-data.mimic_iii_clinical.admissions` a ON p.subject_id = a.subject_id
LEFT JOIN 
    `physionet-data.mimic_iii_clinical.icustays` ie ON a.hadm_id = ie.hadm_id
LEFT JOIN 
    `physionet-data.mimic_iii_clinical.vitalsigns` vs ON ie.icustay_id = vs.icustay_id
WHERE 
    p.gender = 'F'
GROUP BY 
    p.subject_id, p.gender, p.dob, a.hadm_id, a.admittime, a.dischtime, 
    a.admission_type, a.diagnosis, a.hospital_expire_flag
LIMIT 10000
"""

df = client.query(query).to_dataframe()
print(f"Data shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:")
print(df.dtypes)
print(f"\nMortality distribution:")
print(df['mortality'].value_counts())

RefreshError: ('invalid_grant: Bad Request', {'error': 'invalid_grant', 'error_description': 'Bad Request'})

#### Data Exploration and Preprocessing

In [ ]:
# Data cleaning and preprocessing
df_clean = df.copy()

# Drop rows with missing target variable
df_clean = df_clean.dropna(subset=['mortality'])

# Handle missing values - fill numeric columns with median
numeric_cols = ['avg_heart_rate', 'avg_sys_bp', 'avg_dias_bp', 'avg_temp', 'avg_resp_rate', 'num_icu_stays']
for col in numeric_cols:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Convert admission_type to numeric
admission_type_map = {v: i for i, v in enumerate(df_clean['admission_type'].unique())}
df_clean['admission_type_code'] = df_clean['admission_type'].map(admission_type_map)

# Calculate age at admission
df_clean['admittime'] = pd.to_datetime(df_clean['admittime'])
df_clean['dob'] = pd.to_datetime(df_clean['dob'])
df_clean['age'] = (df_clean['admittime'] - df_clean['dob']).dt.days / 365.25

# Select features for modeling
features = ['age', 'avg_heart_rate', 'avg_sys_bp', 'avg_dias_bp', 'avg_temp', 'avg_resp_rate', 'num_icu_stays', 'admission_type_code']
X = df_clean[features].copy()
y = df_clean['mortality'].astype(int)

# Remove any remaining NaN values
mask = ~(X.isna().any(axis=1))
X = X[mask]
y = y[mask]

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nMortality rate: {y.mean():.2%}")
print(f"\nFeature statistics:")
print(X.describe())

#### Model Training and Evaluation

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression model
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

# Train random forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train_scaled, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

# Evaluate Logistic Regression
print("=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_lr):.4f}")

# Evaluate Random Forest
print("\n" + "=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_rf):.4f}")

#### Feature Importance and Visualization

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance (Random Forest):")
print(feature_importance)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance for Mortality Prediction')
plt.tight_layout()
plt.show()

# Plot ROC curves
plt.figure(figsize=(10, 6))
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)

plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={roc_auc_score(y_test, y_pred_proba_lr):.4f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, y_pred_proba_rf):.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Women Mortality Prediction')
plt.legend()
plt.tight_layout()
plt.show()

# Confusion matrix visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', ax=axes[0], cmap='Blues')
axes[0].set_title('Logistic Regression Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', ax=axes[1], cmap='Blues')
axes[1].set_title('Random Forest Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')
plt.tight_layout()
plt.show()